In [71]:
# --- Cell 1: Load and preprocess images ---
import cv2
import os
import numpy as np

data_path = "/content/drive/MyDrive/Brain_Tumor"
categories = os.listdir(data_path)
print(categories)

labels = [i for i in range(len(categories))]
label_dict = dict(zip(categories, labels))
print(label_dict)

img_size = 128          # bumped up from 100 -> gives ResNet more spatial detail to work with
data = []
target = []
skipped = 0

for category in categories:
    folder_path = os.path.join(data_path, category)
    image_names = os.listdir(folder_path)

    for img_name in image_names:
        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path)

        if img is None:                       # FIX: catch unreadable files explicitly
            skipped += 1
            continue

        try:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            resized = cv2.resize(img_rgb, (img_size, img_size))
            data.append(resized)
            target.append(label_dict[category])
        except Exception as e:
            skipped += 1
            print("Exception:", e)

print(f"Loaded {len(data)} images, skipped {skipped} unreadable files")

['yes', 'no']
{'yes': 0, 'no': 1}
Loaded 253 images, skipped 0 unreadable files


In [72]:
# --- Cell 2: Check class balance (new — worth knowing before you trust "accuracy") ---
import collections
print("Class distribution:", collections.Counter(target))

Class distribution: Counter({0: 155, 1: 98})


In [73]:
# --- Cell 3: Arrays + proper 3-way split (train / val / test) ---
data = np.array(data)                          # keep uint8 0-255 here — preprocess_input handles scaling
target = np.array(target)

from tensorflow.keras.utils import to_categorical
new_target = to_categorical(target)

from sklearn.model_selection import train_test_split

# FIX: split into train / val / test instead of reusing the test set as validation
x_train, x_temp, y_train, y_temp = train_test_split(
    data, new_target, test_size=0.2, random_state=42, stratify=target
)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42,
    stratify=np.argmax(y_temp, axis=1)
)
print("Train:", x_train.shape, "Val:", x_val.shape, "Test:", x_test.shape)

Train: (202, 128, 128, 3) Val: (25, 128, 128, 3) Test: (26, 128, 128, 3)


In [74]:
# --- Cell 4: Build the model ---
from tensorflow.keras.layers import (
    RandomFlip, RandomRotation, RandomZoom, RandomTranslation, RandomContrast,
    Dense, Dropout, Input, GlobalAveragePooling2D, Lambda
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

base_model = ResNet50(weights='imagenet', include_top=False,
                       input_shape=(img_size, img_size, 3))
base_model.trainable = False   # phase 1: frozen backbone

model = Sequential([
    Input(shape=(img_size, img_size, 3)),

    # Augmentation (training-time only)
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomTranslation(height_factor=0.08, width_factor=0.08),
    RandomZoom(0.08),
    RandomContrast(0.15),

    # FIX: correct ResNet50 preprocessing instead of manual /255.0
    Lambda(preprocess_input),

    base_model,
    GlobalAveragePooling2D(),

    Dropout(0.5),
    Dense(50, activation='relu'),
    Dense(2, activation='softmax')
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip_4 (RandomFlip)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_4               │ (None, 128, 128, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation_4            │ (None, 128, 128, 3)    │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_4 (RandomZoom)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast_4               │ (None, 128, 128, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 50)             │       102,450 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 2)              │           102 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,690,264 (90.37 MB)

 Trainable params: 102,552 (400.59 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [75]:
# --- Cell 5: Phase 1 training — frozen backbone ---
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

history_phase1 = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),      # FIX: validation set, not test set
    epochs=50,
    callbacks=[early_stop]
)

Epoch 1/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 12s 639ms/step - accuracy: 0.6139 - loss: 0.9822 - val_accuracy: 0.5200 - val_loss: 0.6764
Epoch 2/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step - accuracy: 0.7228 - loss: 0.5381 - val_accuracy: 0.8400 - val_loss: 0.4342
Epoch 3/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 86ms/step - accuracy: 0.8020 - loss: 0.4842 - val_accuracy: 0.8800 - val_loss: 0.3468
Epoch 4/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step - accuracy: 0.8713 - loss: 0.3303 - val_accuracy: 0.8800 - val_loss: 0.2923
Epoch 5/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 88ms/step - accuracy: 0.8614 - loss: 0.3619 - val_accuracy: 0.9200 - val_loss: 0.2662
Epoch 6/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 85ms/step - accuracy: 0.8564 - loss: 0.3240 - val_accuracy: 0.9600 - val_loss: 0.2554
Epoch 7/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 86ms/step - accuracy: 0.8564 - loss: 0.2918 - val_accuracy: 0.9600 - val_loss: 0.2362
Epoch 8/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step - accuracy: 0.9059 - loss: 0.2546 - val_accuracy: 0.9200 - val_loss: 0.234

In [76]:
# --- Cell 6: Phase 2 — fine-tune the top of ResNet50 ---
# Unfreeze the last few blocks and train with a low learning rate.
base_model.trainable = True

# Freeze everything except roughly the last 20 layers of ResNet50
fine_tune_at = len(base_model.layers) - 20
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

from tensorflow.keras.optimizers import Adam

model.compile(optimizer=Adam(learning_rate=1e-5),   # much lower LR for fine-tuning
              loss="categorical_crossentropy", metrics=["accuracy"])

early_stop_ft = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

history_phase2 = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=30,
    callbacks=[early_stop_ft]
)

Epoch 1/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 14s 518ms/step - accuracy: 0.8713 - loss: 0.3239 - val_accuracy: 0.9600 - val_loss: 0.1988
Epoch 2/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 86ms/step - accuracy: 0.9109 - loss: 0.2653 - val_accuracy: 0.9600 - val_loss: 0.2004
Epoch 3/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 78ms/step - accuracy: 0.8812 - loss: 0.3089 - val_accuracy: 0.9600 - val_loss: 0.2020
Epoch 4/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - accuracy: 0.8812 - loss: 0.2842 - val_accuracy: 0.9600 - val_loss: 0.2064
Epoch 5/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - accuracy: 0.8812 - loss: 0.2472 - val_accuracy: 0.9200 - val_loss: 0.2115
Epoch 6/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - accuracy: 0.8762 - loss: 0.2658 - val_accuracy: 0.8800 - val_loss: 0.2163
Epoch 7/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 79ms/step - accuracy: 0.9059 - loss: 0.2588 - val_accuracy: 0.8800 - val_loss: 0.2160
Epoch 8/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - accuracy: 0.9257 - loss: 0.1986 - val_accuracy: 0.8800 - val_loss: 0.217

In [77]:
# --- Cell 7: Final evaluation on the untouched test set ---
loss, accuracy = model.evaluate(x_test, y_test)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Optional: confusion matrix / precision / recall for a fuller picture than accuracy alone
from sklearn.metrics import classification_report, confusion_matrix

y_pred = np.argmax(model.predict(x_test), axis=1)
y_true = np.argmax(y_test, axis=1)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=list(label_dict.keys())))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 381ms/step - accuracy: 0.8846 - loss: 0.4392
Test Loss: 0.4392
Test Accuracy: 0.8846


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
[[16  0]
 [ 3  7]]
              precision    recall  f1-score   support

         yes       0.84      1.00      0.91        16
          no       1.00      0.70      0.82        10

    accuracy                           0.88        26
   macro avg       0.92      0.85      0.87        26
weighted avg       0.90      0.88      0.88        26

